# 🧪 Exploratory Data Analysis — Polymarket RL Project

**Team:** Fatima PAGKALIWANGAN · Jose ROMERO MOLINA · Yurong WEN

This notebook covers:
1. Raw data inspection
2. Price / volume distributions
3. Market resolution statistics
4. Feature engineering validation
5. 80/20 temporal weighting visualisation
6. Sentiment score distribution

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "figure.figsize": (12, 4)})

RAW_PATH       = "../data/raw/polymarket_raw.parquet"
PROCESSED_PATH = "../data/processed/features.parquet"
SENTIMENT_PATH = "../data/sentiment/sentiment_scores.parquet"

print("Imports OK ✓")

## 1 · Raw Data Inspection

In [ ]:
df_raw = pd.read_parquet(RAW_PATH)
print(f"Shape: {df_raw.shape}")
print(f"Markets: {df_raw['market_id'].nunique()}")
print(f"Date range: {df_raw['t'].min()} → {df_raw['t'].max()}")
display(df_raw.head())
display(df_raw.dtypes.to_frame("dtype"))

## 2 · YES Price & Volume Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# YES price distribution
axes[0].hist(df_raw["c"].clip(0, 1), bins=50, color="#2563eb", alpha=0.8)
axes[0].set_title("YES Price Distribution")
axes[0].set_xlabel("Price ($)")
axes[0].set_ylabel("Count")
axes[0].axvline(0.5, color="red", linestyle="--", label="0.50")
axes[0].legend()

# Volume distribution (log scale)
vol = df_raw["volume"].replace(0, np.nan).dropna()
axes[1].hist(np.log1p(vol), bins=50, color="#16a34a", alpha=0.8)
axes[1].set_title("log(1 + Volume) Distribution")
axes[1].set_xlabel("log(1 + Volume)")

# Candles per market
candles_per_market = df_raw.groupby("market_id").size()
axes[2].hist(candles_per_market, bins=40, color="#d97706", alpha=0.8)
axes[2].set_title("Candles per Market")
axes[2].set_xlabel("# Hourly candles")
axes[2].axvline(candles_per_market.mean(), color="red", linestyle="--",
                label=f"mean={candles_per_market.mean():.0f}")
axes[2].legend()

plt.tight_layout()
plt.savefig("../results/eda_price_volume.png", dpi=130, bbox_inches="tight")
plt.show()

## 3 · Market Resolution Statistics

In [ ]:
outcomes = df_raw.groupby("market_id")["outcome"].first()
print(outcomes.value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = outcomes.value_counts()
axes[0].pie(
    counts.values,
    labels=["YES resolved", "NO resolved"],
    autopct="%1.1f%%",
    colors=["#16a34a", "#dc2626"],
    startangle=90,
)
axes[0].set_title("Market Outcome Distribution")

# Final price at resolution vs outcome
last_price = df_raw.groupby("market_id").apply(lambda g: g.sort_values("t").iloc[-1]["c"])
meta = pd.DataFrame({"last_price": last_price, "outcome": outcomes}).dropna()
yes_prices = meta[meta["outcome"] == 1]["last_price"]
no_prices  = meta[meta["outcome"] == 0]["last_price"]

axes[1].hist(yes_prices, bins=30, alpha=0.7, color="#16a34a", label="YES markets")
axes[1].hist(no_prices,  bins=30, alpha=0.7, color="#dc2626", label="NO markets")
axes[1].set_title("Final YES Price by Outcome")
axes[1].set_xlabel("Final YES price")
axes[1].legend()

plt.tight_layout()
plt.savefig("../results/eda_outcomes.png", dpi=130, bbox_inches="tight")
plt.show()

## 4 · Feature Engineering Validation

In [ ]:
from data.preprocessing import add_technical_features, FEATURE_COLS

df_feat = add_technical_features(df_raw.copy())
df_feat = df_feat.dropna(subset=FEATURE_COLS)
print(f"After feature engineering: {df_feat.shape}")
display(df_feat[FEATURE_COLS].describe().round(4))

# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df_feat[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../results/eda_correlation.png", dpi=130, bbox_inches="tight")
plt.show()

## 5 · 80/20 Temporal Weighting Visualisation

In [ ]:
from data.preprocessing import compute_temporal_weights

n   = 500
w   = compute_temporal_weights(n, recent_weight=0.80, old_weight=0.20)
cum = np.cumsum(w)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Per-row weight
axes[0].plot(w * n, color="#2563eb")               # scale up for readability
axes[0].axvline(n // 2, color="red", linestyle="--", label="50% time split")
axes[0].fill_between(range(n // 2),           w[:n//2] * n, alpha=0.2, color="#dc2626", label="Old  (20%)")
axes[0].fill_between(range(n // 2, n), w[n//2:] * n, alpha=0.2, color="#16a34a", label="Recent (80%)")
axes[0].set_title("Per-Row Sample Weight")
axes[0].set_xlabel("Timestep index")
axes[0].set_ylabel("Relative weight")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Cumulative weight
axes[1].plot(cum, color="#2563eb")
axes[1].axvline(n // 2, color="red", linestyle="--", label="50% time split")
axes[1].axhline(0.20, color="#dc2626", linestyle=":", label="20% total weight")
axes[1].axhline(1.00, color="#16a34a", linestyle=":", label="100% total weight")
axes[1].set_title("Cumulative Weight")
axes[1].set_xlabel("Timestep index")
axes[1].set_ylabel("Cumulative weight")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("80 / 20 Temporal Weighting Scheme", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/eda_temporal_weights.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"Weight sum (should be 1.0): {w.sum():.6f}")
print(f"Old half  weight: {w[:n//2].sum():.3f}  | Recent half weight: {w[n//2:].sum():.3f}")

## 6 · Environment Sanity Check

In [ ]:
from env.polymarket_env import PolymarketEnv

# Use first market from feature data
mid0  = df_feat["market_id"].iloc[0]
mkt0  = df_feat[df_feat["market_id"] == mid0]
feat0 = mkt0[FEATURE_COLS].values.astype("float32")
out0  = int(mkt0["outcome"].iloc[0])

env = PolymarketEnv(feat0, out0, initial_cash=1000.0, render_mode="human")
obs, _ = env.reset()
print(f"Obs shape: {obs.shape}  |  Action space: {env.action_space}")

# Random rollout
np.random.seed(7)
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

print(f"\nEpisode done.")
print(f"  Total return  : {env.total_return():+.2f}%")
print(f"  Sharpe ratio  : {env.sharpe_ratio():.3f}")
print(f"  Max drawdown  : {env.max_drawdown():+.2f}%")

plt.figure(figsize=(12, 3))
plt.plot(env.portfolio_history, color="#2563eb")
plt.axhline(1000, color="grey", linestyle="--")
plt.title("Random Agent — Portfolio Value (sanity check)")
plt.xlabel("Step"); plt.ylabel("Portfolio ($)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()